# BPR-MF validation sanity — fixed configuration

Đây là trained baseline đầu tiên. Notebook chỉ trả lời: một mô hình latent-factor học từ toàn bộ training edge có vượt failure case MostPop khi dùng đúng cùng validation population, strict-history mask và full-catalog evaluator hay không?

Ranh giới:

- một configuration đã đăng ký trước trong `configs/bpr_mf_sanity_v1.json`;
- training trên toàn bộ 3.868.654 edge;
- uniform negative sampling nhưng loại mọi training-positive của user;
- exact full-catalog NDCG@20/Recall@20 và deterministic rank tie-break;
- chỉ đánh giá validation, tuyệt đối không đọc test;
- không hyperparameter search, không claim final baseline hoặc scalability.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không ở Colab; notebook này cần artifact tương đương và GPU CUDA.')

DRIVE_ROOT = Path('/content/drive/MyDrive/Phase2_Amazon_Audit')
GRAPH_DIR = DRIVE_ROOT / 'g2c_baby_p4'
MOSTPOP_DIR = DRIVE_ROOT / 'mostpop_validation'
OUTPUT_DIR = DRIVE_ROOT / 'bpr_mf_sanity_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = GRAPH_DIR / 'baby_p4_g2c_manifest.json'
MOSTPOP_PATH = MOSTPOP_DIR / 'mostpop_validation_summary.json'

for required in (MANIFEST_PATH, MOSTPOP_PATH):
    if not required.exists():
        raise FileNotFoundError(f'Thiếu artifact: {required}')

print('Manifest:', MANIFEST_PATH)
print('MostPop control:', MOSTPOP_PATH)
print('Output:', OUTPUT_DIR)


In [ ]:
REGISTERED_CONFIG_FILE_SHA256 = 'f14ff370fbf669b774374fe263ddb63cedd2dd35a0fbc55194f87fd606b14900'

CONFIG = {
    'config_id': 'bpr-mf-sanity-v1-2026-09-13',
    'model': {
        'name': 'BPR-MF',
        'embedding_dim': 64,
        'initialization_std': 0.01,
        'bias': False,
    },
    'training': {
        'seed': 20260913,
        'epochs': 5,
        'batch_size': 65536,
        'optimizer': 'SparseAdam',
        'learning_rate': 0.01,
        'l2_coefficient': 1e-6,
        'negative_sampling': 'uniform_training_catalog_reject_all_training_positives',
        'epoch_selection': 'fixed_last_epoch',
        'mixed_precision': False,
    },
    'evaluation': {
        'split': 'validation_only',
        'k': 20,
        'eval_batch_size': 128,
        'candidate_universe': 'all frozen training items minus strict prior mapped positives',
        'timestamp_rule': 'events at the target timestamp are not prior history',
        'rank_tie_break': 'item_idx_ascending',
        'target_unit': 'one relevant item per target row',
    },
    'claim_boundary': (
        'One fixed validation-only sanity run. No test access, hyperparameter search, '
        'final baseline, sampler comparison, or scalability claim.'
    ),
}

assert CONFIG['evaluation']['split'] == 'validation_only'
assert CONFIG['training']['epoch_selection'] == 'fixed_last_epoch'
assert CONFIG['training']['epochs'] == 5
print('Frozen config:', CONFIG['config_id'])
print('Registered file SHA-256:', REGISTERED_CONFIG_FILE_SHA256)


## Gate logic

MostPop đã chứng minh data/evaluator invariant. BPR-MF giữ nguyên population và evaluator, chỉ thay scoring function.

Một target được xếp hạng trước item có score thấp hơn. Khi score bằng nhau, `item_idx` nhỏ hơn đứng trước. Recommendation top-20 cũng xử lý tie ở boundary bằng `item_idx`, không dựa vào thứ tự không ổn định của GPU.

BPR-MF phải hoàn thành toàn bộ training epoch đã đăng ký. Test target không được đọc.


In [ ]:
from collections import Counter, defaultdict
from itertools import groupby
from math import log2
import csv
import gc
import gzip
import hashlib
import json
import os
import platform
import resource
import time

import numpy as np


MOSTPOP_SUMMARY_SHA256 = '00332090cae73a563a7fcafde895572fb06d574366ce37bf6f9d90208dfccf62'


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_artifact(entry, fallback_name):
    recorded = Path(entry['path'])
    fallback = GRAPH_DIR / fallback_name
    if recorded.exists():
        return recorded
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f"Không tìm thấy artifact: {recorded} hoặc {fallback}")


def row_metrics(ranks, k):
    ranks = np.asarray(ranks, dtype=np.int64)
    if ranks.size == 0:
        return {'rows': 0, 'hits_at_k': 0, 'recall_at_k': None, 'ndcg_at_k': None, 'k': k}
    hits = ranks <= k
    discounts = np.zeros(ranks.size, dtype=np.float64)
    discounts[hits] = 1.0 / np.log2(ranks[hits] + 1.0)
    return {
        'rows': int(ranks.size),
        'hits_at_k': int(hits.sum()),
        'recall_at_k': float(hits.mean()),
        'ndcg_at_k': float(discounts.mean()),
        'k': int(k),
    }


def item_cohort(degree):
    if degree >= 397:
        return 'head'
    if degree >= 13:
        return 'body'
    return 'tail'


def user_cohort(degree):
    if degree == 1:
        return 'singleton'
    if degree <= 3:
        return 'repeat_light'
    return 'active'


def grouped_metrics(ranks, labels, k):
    ranks = np.asarray(ranks)
    labels = np.asarray(labels)
    result = {}
    for label in sorted(set(labels.tolist())):
        mask = labels == label
        result[label] = {
            **row_metrics(ranks[mask], k),
            'target_share': float(mask.mean()),
        }
    return result


def process_peak_rss_mb():
    value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return float(value / 1024.0)


def positive_collision_mask(users, items, positive_keys, item_count):
    keys = users.astype(np.int64, copy=False) * item_count + items.astype(np.int64, copy=False)
    positions = np.searchsorted(positive_keys, keys)
    in_bounds = positions < positive_keys.size
    collisions = np.zeros(keys.size, dtype=bool)
    collisions[in_bounds] = positive_keys[positions[in_bounds]] == keys[in_bounds]
    return collisions


def sample_exact_uniform_negatives(users, rng, positive_keys, item_count):
    negatives = rng.integers(0, item_count, size=users.size, dtype=np.int32)
    collisions = positive_collision_mask(users, negatives, positive_keys, item_count)
    resampled = 0
    rounds = 0
    while collisions.any():
        count = int(collisions.sum())
        negatives[collisions] = rng.integers(0, item_count, size=count, dtype=np.int32)
        resampled += count
        rounds += 1
        if rounds > 100:
            raise RuntimeError('Negative rejection sampler không hội tụ')
        collisions = positive_collision_mask(users, negatives, positive_keys, item_count)
    return negatives, resampled


In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
mostpop = json.loads(MOSTPOP_PATH.read_text(encoding='utf-8'))

if sha256_file(MOSTPOP_PATH) != MOSTPOP_SUMMARY_SHA256:
    raise AssertionError('MostPop summary không đúng artifact đã review')
if mostpop['status'] != 'MOSTPOP_VALIDATION_SANITY_EXECUTED':
    raise AssertionError('MostPop control chưa hoàn thành')
if not all(mostpop['integrity_assertions'].values()):
    raise AssertionError('MostPop control có integrity assertion fail')

user_count = int(manifest['training_graph']['users'])
item_count = int(manifest['training_graph']['items'])
train_entry = manifest['artifacts']['train_edges']
validation_entry = manifest['artifacts']['validation_targets']
train_path = resolve_artifact(train_entry, 'baby_p4_train_edges.csv.gz')
validation_path = resolve_artifact(validation_entry, 'baby_p4_validation_targets.csv.gz')

print('1/6 Checksum...')
train_sha = sha256_file(train_path)
validation_sha = sha256_file(validation_path)
if train_sha != train_entry['sha256']:
    raise AssertionError('SHA-256 train_edges không khớp manifest')
if validation_sha != validation_entry['sha256']:
    raise AssertionError('SHA-256 validation_targets không khớp manifest')

validation_rows_expected = int(validation_entry['rows'])
target_users = np.empty(validation_rows_expected, dtype=np.int32)
target_items = np.empty(validation_rows_expected, dtype=np.int32)
target_timestamps = np.empty(validation_rows_expected, dtype=np.int64)
target_source_rows = np.empty(validation_rows_expected, dtype=np.int64)
target_candidate_counts = np.empty(validation_rows_expected, dtype=np.int32)

print('2/6 Đọc validation target...')
with gzip.open(validation_path, 'rt', encoding='utf-8', newline='') as handle:
    for index, row in enumerate(csv.DictReader(handle)):
        if index >= validation_rows_expected:
            raise AssertionError('Validation artifact có nhiều row hơn manifest')
        target_users[index] = int(row['user_idx'])
        target_items[index] = int(row['item_idx'])
        target_timestamps[index] = int(row['timestamp_ms'])
        target_source_rows[index] = int(row['source_row'])
        target_candidate_counts[index] = int(row['candidate_count'])
        validation_rows = index + 1
if validation_rows != validation_rows_expected:
    raise AssertionError('Validation row count không khớp manifest')

eval_user_mask = np.zeros(user_count, dtype=bool)
eval_user_mask[np.unique(target_users)] = True
base_history = defaultdict(list)

train_rows_expected = int(train_entry['rows'])
train_users = np.empty(train_rows_expected, dtype=np.int32)
train_items = np.empty(train_rows_expected, dtype=np.int32)

print('3/6 Đọc training edge...')
with gzip.open(train_path, 'rt', encoding='utf-8', newline='') as handle:
    for index, row in enumerate(csv.DictReader(handle)):
        if index >= train_rows_expected:
            raise AssertionError('Training artifact có nhiều row hơn manifest')
        user_idx = int(row['user_idx'])
        item_idx = int(row['item_idx'])
        train_users[index] = user_idx
        train_items[index] = item_idx
        if eval_user_mask[user_idx]:
            base_history[user_idx].append(item_idx)
        train_rows = index + 1
if train_rows != train_rows_expected:
    raise AssertionError('Training row count không khớp manifest')

print('4/6 Degree và positive-key index...')
user_degrees = np.bincount(train_users, minlength=user_count)
item_degrees = np.bincount(train_items, minlength=item_count)
if int(user_degrees.sum()) != train_rows or int(item_degrees.sum()) != train_rows:
    raise AssertionError('Degree mass không bằng training rows')

positive_keys = train_users.astype(np.int64)
positive_keys *= item_count
positive_keys += train_items
positive_keys.sort()
if np.any(np.diff(positive_keys) == 0):
    raise AssertionError('Training user-item pair không unique')

print('5/6 Dựng strict prior history cho từng validation target...')
targets_by_user = defaultdict(list)
for target_index, user_idx in enumerate(target_users):
    targets_by_user[int(user_idx)].append(target_index)

histories = [None] * validation_rows
candidate_checks = 0
target_not_in_prior = True

for user_idx, indices in targets_by_user.items():
    prior = set(base_history[user_idx])
    indices.sort(key=lambda index: (int(target_timestamps[index]), int(target_source_rows[index])))
    for _, timestamp_group in groupby(indices, key=lambda index: int(target_timestamps[index])):
        group = list(timestamp_group)
        for target_index in group:
            target_item = int(target_items[target_index])
            if target_item in prior:
                target_not_in_prior = False
                raise AssertionError(f'Target đã nằm trong prior history: source_row={target_source_rows[target_index]}')
            actual_candidates = item_count - len(prior)
            if actual_candidates != int(target_candidate_counts[target_index]):
                raise AssertionError(
                    f"Candidate count lệch tại source_row={target_source_rows[target_index]}: "
                    f"{actual_candidates} != {target_candidate_counts[target_index]}"
                )
            histories[target_index] = np.asarray(sorted(prior), dtype=np.int32)
            candidate_checks += 1
        prior.update(int(target_items[index]) for index in group)

if any(history is None for history in histories):
    raise AssertionError('Có validation target thiếu history')

print('6/6 Negative sampler preflight...')
preflight_rng = np.random.default_rng(CONFIG['training']['seed'])
preflight_size = min(50000, train_rows)
preflight_indices = preflight_rng.choice(train_rows, size=preflight_size, replace=False)
preflight_users = train_users[preflight_indices]
preflight_negatives, preflight_resamples = sample_exact_uniform_negatives(
    preflight_users, preflight_rng, positive_keys, item_count
)
if positive_collision_mask(preflight_users, preflight_negatives, positive_keys, item_count).any():
    raise AssertionError('Negative sampler còn training-positive collision')

source_assertions = {
    'train_sha256_matches_manifest': train_sha == train_entry['sha256'],
    'validation_sha256_matches_manifest': validation_sha == validation_entry['sha256'],
    'train_rows_match_manifest': train_rows == train_rows_expected,
    'validation_rows_match_manifest': validation_rows == validation_rows_expected,
    'degree_mass_matches_train_rows': int(item_degrees.sum()) == train_rows == int(user_degrees.sum()),
    'training_pairs_unique': not np.any(np.diff(positive_keys) == 0),
    'candidate_count_matches_every_target': candidate_checks == validation_rows,
    'target_not_in_strict_prior_history': target_not_in_prior,
    'negative_sampler_preflight_has_no_positive_collision': True,
    'mostpop_control_integrity_passed': all(mostpop['integrity_assertions'].values()),
    'test_targets_not_read': True,
}
if not all(source_assertions.values()):
    raise AssertionError(source_assertions)

print(json.dumps({
    'source_assertions': source_assertions,
    'train_rows': train_rows,
    'validation_rows': validation_rows,
    'validation_users': int(np.unique(target_users).size),
    'negative_preflight_resamples': preflight_resamples,
    'process_peak_rss_mb': process_peak_rss_mb(),
}, indent=2))


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError('BPR-MF sanity run yêu cầu Colab GPU CUDA. Chọn Runtime → Change runtime type → GPU.')

device = torch.device('cuda')
torch.manual_seed(CONFIG['training']['seed'])
torch.cuda.manual_seed_all(CONFIG['training']['seed'])
torch.backends.cuda.matmul.allow_tf32 = False


class BPRMF(nn.Module):
    def __init__(self, users, items, embedding_dim, init_std):
        super().__init__()
        self.user = nn.Embedding(users, embedding_dim, sparse=True)
        self.item = nn.Embedding(items, embedding_dim, sparse=True)
        nn.init.normal_(self.user.weight, std=init_std)
        nn.init.normal_(self.item.weight, std=init_std)

    def triplet_scores(self, users, positives, negatives):
        user = self.user(users)
        positive = self.item(positives)
        negative = self.item(negatives)
        return user, positive, negative


model = BPRMF(
    user_count,
    item_count,
    CONFIG['model']['embedding_dim'],
    CONFIG['model']['initialization_std'],
).to(device)

initial_item_probe = model.item.weight[:1024].detach().cpu().clone()
optimizer = torch.optim.SparseAdam(
    model.parameters(),
    lr=CONFIG['training']['learning_rate'],
)

torch.cuda.reset_peak_memory_stats(device)
training_started = time.perf_counter()
epoch_records = []
total_negative_resamples = 0
finite_losses = True

for epoch in range(CONFIG['training']['epochs']):
    epoch_started = time.perf_counter()
    epoch_rng = np.random.default_rng(CONFIG['training']['seed'] + epoch)
    permutation = epoch_rng.permutation(train_rows)
    loss_sum = 0.0
    example_count = 0

    model.train()
    for start in range(0, train_rows, CONFIG['training']['batch_size']):
        batch_indices = permutation[start:start + CONFIG['training']['batch_size']]
        users_np = train_users[batch_indices]
        positives_np = train_items[batch_indices]
        negatives_np, resampled = sample_exact_uniform_negatives(
            users_np, epoch_rng, positive_keys, item_count
        )
        total_negative_resamples += resampled

        users = torch.from_numpy(users_np).to(device=device, dtype=torch.long)
        positives = torch.from_numpy(positives_np).to(device=device, dtype=torch.long)
        negatives = torch.from_numpy(negatives_np).to(device=device, dtype=torch.long)

        user_vec, positive_vec, negative_vec = model.triplet_scores(users, positives, negatives)
        positive_score = (user_vec * positive_vec).sum(dim=1)
        negative_score = (user_vec * negative_vec).sum(dim=1)
        ranking_loss = -F.logsigmoid(positive_score - negative_score).mean()
        regularization = (
            user_vec.square().sum(dim=1)
            + positive_vec.square().sum(dim=1)
            + negative_vec.square().sum(dim=1)
        ).mean()
        loss = ranking_loss + CONFIG['training']['l2_coefficient'] * regularization

        if not torch.isfinite(loss):
            finite_losses = False
            raise FloatingPointError(f'Loss không hữu hạn tại epoch {epoch + 1}')

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch_examples = int(users_np.size)
        loss_sum += float(loss.detach().cpu()) * batch_examples
        example_count += batch_examples

    torch.cuda.synchronize(device)
    epoch_record = {
        'epoch': epoch + 1,
        'mean_loss': loss_sum / example_count,
        'examples': example_count,
        'wall_seconds': time.perf_counter() - epoch_started,
    }
    epoch_records.append(epoch_record)
    print(epoch_record)

training_wall_seconds = time.perf_counter() - training_started
training_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
parameters_changed = not torch.equal(initial_item_probe, model.item.weight[:1024].detach().cpu())

del optimizer
gc.collect()
torch.cuda.empty_cache()

training_assertions = {
    'cuda_used': device.type == 'cuda',
    'all_registered_epochs_completed': len(epoch_records) == CONFIG['training']['epochs'],
    'each_epoch_saw_all_training_edges': all(record['examples'] == train_rows for record in epoch_records),
    'losses_finite': finite_losses and all(np.isfinite(record['mean_loss']) for record in epoch_records),
    'item_parameters_changed': parameters_changed,
    'negative_sampler_policy_enforced': True,
    'fixed_last_epoch_used': True,
}
if not all(training_assertions.values()):
    raise AssertionError(training_assertions)

print(json.dumps({
    'training_assertions': training_assertions,
    'epoch_records': epoch_records,
    'negative_resamples': total_negative_resamples,
    'training_wall_seconds': training_wall_seconds,
    'training_peak_gpu_mb': training_peak_gpu_mb,
}, indent=2))


In [ ]:
def resolve_boundary_ties(scores, top_values, top_indices, k):
    cutoff = top_values[:, -1]
    total_at_cutoff = (scores == cutoff[:, None]).sum(dim=1)
    selected_at_cutoff = (top_values == cutoff[:, None]).sum(dim=1)
    boundary_rows = torch.nonzero(total_at_cutoff != selected_at_cutoff, as_tuple=False).flatten()
    if boundary_rows.numel() == 0:
        return top_indices, 0

    resolved = top_indices.clone()
    for row in boundary_rows.tolist():
        row_scores = scores[row]
        row_cutoff = cutoff[row]
        better = torch.nonzero(row_scores > row_cutoff, as_tuple=False).flatten()
        tied = torch.nonzero(row_scores == row_cutoff, as_tuple=False).flatten()
        needed = k - int(better.numel())
        chosen = torch.cat((better, tied[:needed]))
        if chosen.numel() != k:
            raise AssertionError('Không resolve được top-k boundary tie')
        resolved[row] = chosen
    return resolved, int(boundary_rows.numel())


model.eval()
k = CONFIG['evaluation']['k']
eval_batch_size = CONFIG['evaluation']['eval_batch_size']
ranks = np.empty(validation_rows, dtype=np.int32)
recommended_mask = np.zeros(item_count, dtype=bool)
exposure_counts = Counter()
boundary_tie_rows = 0
item_ids = torch.arange(item_count, device=device, dtype=torch.long)
item_matrix = model.item.weight.detach()

torch.cuda.reset_peak_memory_stats(device)
evaluation_started = time.perf_counter()

with torch.no_grad():
    for start in range(0, validation_rows, eval_batch_size):
        stop = min(start + eval_batch_size, validation_rows)
        users = torch.from_numpy(target_users[start:stop]).to(device=device, dtype=torch.long)
        targets = torch.from_numpy(target_items[start:stop]).to(device=device, dtype=torch.long)

        scores = model.user(users) @ item_matrix.T

        mask_rows = []
        mask_items = []
        for local_row, history in enumerate(histories[start:stop]):
            if history.size:
                mask_rows.append(np.full(history.size, local_row, dtype=np.int64))
                mask_items.append(history.astype(np.int64, copy=False))
        if mask_rows:
            row_tensor = torch.from_numpy(np.concatenate(mask_rows)).to(device)
            item_tensor = torch.from_numpy(np.concatenate(mask_items)).to(device)
            scores[row_tensor, item_tensor] = -torch.inf

        row_ids = torch.arange(stop - start, device=device)
        target_scores = scores[row_ids, targets]
        if not torch.isfinite(target_scores).all():
            raise AssertionError('Target score bị mask hoặc không hữu hạn')

        strictly_better = (scores > target_scores[:, None]).sum(dim=1)
        equal_and_lower_id = (
            (scores == target_scores[:, None])
            & (item_ids[None, :] < targets[:, None])
        ).sum(dim=1)
        batch_ranks = 1 + strictly_better + equal_and_lower_id
        ranks[start:stop] = batch_ranks.cpu().numpy().astype(np.int32)

        top_values, top_indices = torch.topk(scores, k=k, dim=1, largest=True, sorted=True)
        top_indices, resolved_rows = resolve_boundary_ties(scores, top_values, top_indices, k)
        boundary_tie_rows += resolved_rows
        top_numpy = top_indices.cpu().numpy()
        recommended_mask[top_numpy.reshape(-1)] = True

        top_degrees = item_degrees[top_numpy.reshape(-1)]
        exposure_counts['head'] += int((top_degrees >= 397).sum())
        exposure_counts['body'] += int(((top_degrees >= 13) & (top_degrees <= 396)).sum())
        exposure_counts['tail'] += int((top_degrees <= 12).sum())

        if (start // eval_batch_size) % 100 == 0:
            print(f'Evaluated {stop:,}/{validation_rows:,} target')

torch.cuda.synchronize(device)
evaluation_wall_seconds = time.perf_counter() - evaluation_started
evaluation_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

overall_metrics = row_metrics(ranks, k)
item_labels = np.asarray([item_cohort(item_degrees[item]) for item in target_items])
user_labels = np.asarray([user_cohort(user_degrees[user]) for user in target_users])
item_cohort_metrics = grouped_metrics(ranks, item_labels, k)
user_cohort_metrics = grouped_metrics(ranks, user_labels, k)

recommendation_slots = validation_rows * k
if sum(exposure_counts.values()) != recommendation_slots:
    raise AssertionError('Exposure count không bằng validation_rows × k')

coverage = float(recommended_mask.mean())
mostpop_metrics = mostpop['metrics']
mostpop_coverage = mostpop['recommendation_exposure']['catalog_coverage_at_k']

evaluation_assertions = {
    'every_validation_target_ranked': int(ranks.size) == validation_rows,
    'all_ranks_within_candidate_count': bool(np.all(ranks >= 1) and np.all(ranks <= target_candidate_counts)),
    'recommendation_slots_reconcile': sum(exposure_counts.values()) == recommendation_slots,
    'catalog_coverage_reconciles': int(recommended_mask.sum()) <= recommendation_slots,
    'test_targets_not_read': True,
    'same_validation_rows_as_mostpop': validation_rows == int(mostpop_metrics['rows']),
    'rank_tie_break_applied': True,
    'topk_boundary_ties_resolved': True,
}
if not all(evaluation_assertions.values()):
    raise AssertionError(evaluation_assertions)

summary = {
    'status': 'BPR_MF_VALIDATION_SANITY_EXECUTED',
    'registered_config': {
        **CONFIG,
        'file_sha256': REGISTERED_CONFIG_FILE_SHA256,
    },
    'source': {
        'manifest_path': str(MANIFEST_PATH),
        'train_edges': {'path': str(train_path), 'rows': train_rows, 'sha256': train_sha},
        'validation_targets': {'path': str(validation_path), 'rows': validation_rows, 'sha256': validation_sha},
        'mostpop_summary': {'path': str(MOSTPOP_PATH), 'sha256': MOSTPOP_SUMMARY_SHA256},
        'users': user_count,
        'items': item_count,
    },
    'integrity_assertions': {
        **source_assertions,
        **training_assertions,
        **evaluation_assertions,
    },
    'training': {
        'epochs': epoch_records,
        'wall_seconds': training_wall_seconds,
        'peak_gpu_memory_mb': training_peak_gpu_mb,
        'negative_resamples': total_negative_resamples,
        'negative_draws': train_rows * CONFIG['training']['epochs'],
    },
    'validation': {
        'metrics': overall_metrics,
        'target_item_cohorts': item_cohort_metrics,
        'user_activity_cohorts': user_cohort_metrics,
        'recommendation_exposure': {
            'recommendation_slots': recommendation_slots,
            'unique_items_at_k': int(recommended_mask.sum()),
            'catalog_coverage_at_k': coverage,
            'item_cohort_share': {
                label: exposure_counts[label] / recommendation_slots
                for label in ('head', 'body', 'tail')
            },
        },
        'wall_seconds': evaluation_wall_seconds,
        'peak_gpu_memory_mb': evaluation_peak_gpu_mb,
        'topk_boundary_tie_rows': boundary_tie_rows,
    },
    'mostpop_comparison': {
        'mostpop_ndcg_at_20': mostpop_metrics['ndcg_at_k'],
        'bpr_mf_ndcg_at_20': overall_metrics['ndcg_at_k'],
        'ndcg_difference': overall_metrics['ndcg_at_k'] - mostpop_metrics['ndcg_at_k'],
        'mostpop_recall_at_20': mostpop_metrics['recall_at_k'],
        'bpr_mf_recall_at_20': overall_metrics['recall_at_k'],
        'recall_difference': overall_metrics['recall_at_k'] - mostpop_metrics['recall_at_k'],
        'mostpop_catalog_coverage_at_20': mostpop_coverage,
        'bpr_mf_catalog_coverage_at_20': coverage,
        'coverage_difference': coverage - mostpop_coverage,
    },
    'environment': {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'gpu': torch.cuda.get_device_name(device),
        'process_peak_rss_mb': process_peak_rss_mb(),
    },
    'claim_boundary': CONFIG['claim_boundary'],
}

print(json.dumps({
    'status': summary['status'],
    'metrics': summary['validation']['metrics'],
    'coverage_at_20': coverage,
    'item_exposure': summary['validation']['recommendation_exposure']['item_cohort_share'],
    'mostpop_comparison': summary['mostpop_comparison'],
    'evaluation_runtime_seconds': evaluation_wall_seconds,
    'evaluation_peak_gpu_mb': evaluation_peak_gpu_mb,
    'all_assertions_pass': all(summary['integrity_assertions'].values()),
}, ensure_ascii=False, indent=2))


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

epochs = [record['epoch'] for record in epoch_records]
losses = [record['mean_loss'] for record in epoch_records]
axes[0].plot(epochs, losses, marker='o', color='#0B0D10')
axes[0].set_title('BPR training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean loss')
axes[0].grid(alpha=0.25)

model_names = ['MostPop', 'BPR-MF']
ndcg_values = [mostpop_metrics['ndcg_at_k'], overall_metrics['ndcg_at_k']]
recall_values = [mostpop_metrics['recall_at_k'], overall_metrics['recall_at_k']]
x = np.arange(2)
width = 0.36
bars_a = axes[1].bar(x - width / 2, ndcg_values, width, label='NDCG@20', color='#3D8DFF')
bars_b = axes[1].bar(x + width / 2, recall_values, width, label='Recall@20', color='#2B9B75')
axes[1].set_xticks(x, model_names)
axes[1].set_title('Validation: cùng evaluator')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].legend(frameon=False)
axes[1].grid(axis='y', alpha=0.25)
for bars in (bars_a, bars_b):
    axes[1].bar_label(bars, labels=[f'{bar.get_height():.2%}' for bar in bars], padding=3)

cohort_order = ['head', 'body', 'tail']
bpr_recall = [item_cohort_metrics[label]['recall_at_k'] for label in cohort_order]
mostpop_recall = [mostpop['target_item_cohorts'][label]['recall_at_k'] for label in cohort_order]
x = np.arange(3)
bars_a = axes[2].bar(x - width / 2, mostpop_recall, width, label='MostPop', color='#D8DEE5')
bars_b = axes[2].bar(x + width / 2, bpr_recall, width, label='BPR-MF', color='#F2A65A')
axes[2].set_xticks(x, ['Head', 'Body', 'Tail'])
axes[2].set_title('Recall@20 theo target item')
axes[2].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[2].legend(frameon=False)
axes[2].grid(axis='y', alpha=0.25)
for bars in (bars_a, bars_b):
    axes[2].bar_label(bars, labels=[f'{bar.get_height():.2%}' for bar in bars], padding=3)

for axis in axes:
    axis.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
figure_path = OUTPUT_DIR / '01_bpr_mf_validation_sanity.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
import zipfile

summary_path = OUTPUT_DIR / 'bpr_mf_validation_summary.json'
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)

def pct(value):
    return f'{value:.2%}'

lines = [
    '# BPR-MF validation sanity',
    '',
    f"- Config: {CONFIG['config_id']}",
    f"- Epochs: {CONFIG['training']['epochs']} fixed",
    f"- NDCG@20: {overall_metrics['ndcg_at_k']:.6f}",
    f"- Recall@20: {overall_metrics['recall_at_k']:.6f}",
    f"- Catalog Coverage@20: {coverage:.6f}",
    f"- ΔNDCG@20 vs MostPop: {summary['mostpop_comparison']['ndcg_difference']:+.6f}",
    f"- ΔRecall@20 vs MostPop: {summary['mostpop_comparison']['recall_difference']:+.6f}",
    '',
    '## Validation theo target item cohort',
    '',
    '| Cohort | Target share | NDCG@20 | Recall@20 |',
    '|---|---:|---:|---:|',
]
for label in ('head', 'body', 'tail'):
    values = item_cohort_metrics[label]
    lines.append(
        f"| {label} | {pct(values['target_share'])} | "
        f"{values['ndcg_at_k']:.6f} | {values['recall_at_k']:.6f} |"
    )
lines.extend([
    '',
    '## Exposure',
    '',
    f"- Unique item@20: {int(recommended_mask.sum()):,}/{item_count:,}",
    f"- Head/body/tail share: "
    f"{pct(exposure_counts['head'] / recommendation_slots)} / "
    f"{pct(exposure_counts['body'] / recommendation_slots)} / "
    f"{pct(exposure_counts['tail'] / recommendation_slots)}",
    '',
    '## Ranh giới claim',
    '',
    summary['claim_boundary'],
    '',
])

report_path = OUTPUT_DIR / 'BPR_MF_VALIDATION_vn.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')

bundle_path = OUTPUT_DIR / 'bpr_mf_validation_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (summary_path, report_path, figure_path):
        archive.write(path, arcname=path.name)

print('Đã tạo:', bundle_path)
print('Dung lượng:', bundle_path.stat().st_size, 'bytes')


## Sau khi chạy

1. Cell nguồn, training và evaluation đều phải có toàn bộ assertion bằng `true`.
2. Xem loss có hữu hạn; không tự tăng epoch hoặc đổi learning rate.
3. Không mở test target.
4. Tải nguyên `bpr_mf_validation_bundle.zip` và gửi lại cho Codex.
5. Codex sẽ đọc failure theo cohort rồi mới quyết định có đủ điều kiện sang full LightGCN hay phải sửa BPR/evaluator.

Nếu CUDA OOM, gửi nguyên traceback và tên GPU. Không giảm embedding dimension hoặc catalog trước khi đăng ký revision cấu hình.


In [ ]:
try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print('Bundle nằm tại:', bundle_path)
